<a href="https://colab.research.google.com/github/chamakov/kaggle-pkm-agent/blob/main/train_in_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [23]:
%%writefile src/rl/train.py
import os
import sys

# Si corre dentro de docker o similar, aseguramos rutas
sys.path.append(os.path.abspath(os.path.join(os.path.dirname(__file__), "../..")))

from src.rl.env_wrapper import CabtGymEnv

try:
    from stable_baselines3 import PPO
    from stable_baselines3.common.callbacks import CheckpointCallback
    has_sb3 = True
except ImportError:
    has_sb3 = False

def train_model():
    print("Inicializando entorno de entrenamiento vs Random...")
    env = CabtGymEnv(opponent_agent="random")

    if has_sb3:
        model = PPO("MultiInputPolicy", env, verbose=1)

        # GUARDADO AUTOMÁTICO: Guarda el modelo cada 10,000 pasos
        checkpoint_callback = CheckpointCallback(
            save_freq=10000,
            save_path='./',
            name_prefix='ppo_cabt_model'
        )

        print("Empezando el entrenamiento... (Autoguardado cada 10,000 pasos)")
        try:
            # El callback guardará archivos como ppo_cabt_model_10000_steps.zip
            model.learn(total_timesteps=1000000000, callback=checkpoint_callback)
        except KeyboardInterrupt:
            print("\nEntrenamiento detenido manualmente.")
    else:
        print("StableBaselines3 no detectado.")

if __name__ == "__main__":
    train_model()

Overwriting src/rl/train.py


In [5]:
!git clone https://github.com/chamakov/kaggle-pkm-agent.git
%cd kaggle-pkm-agent

Cloning into 'kaggle-pkm-agent'...
remote: Enumerating objects: 111, done.
remote: Counting objects: 100% (111/111), done.
remote: Compressing objects: 100% (97/97), done.
remote: Total 111 (delta 10), reused 111 (delta 10), pack-reused 0 (from 0)
Receiving objects: 100% (111/111), 28.11 MiB | 17.89 MiB/s, done.
Resolving deltas: 100% (10/10), done.
/content/kaggle-pkm-agent


# Entrenamiento de CABT con StableBaselines3 en Google Colab

Dado que Colab utiliza Linux nativamente (y tiene GPUs gratuitas), podemos entrenar sin Docker. Sube todos los archivos de tu proyecto a Colab y corre estas celdas.

In [6]:
!pip install --quiet kaggle-environments stable-baselines3 shimmy

## Inyectar el Motor Avanzado (C++)
El siguiente comando sobreescribe el motor de C++ básico que viene por defecto en Kaggle Environments, inyectando el motor avanzado que tenemos en la carpeta `remotethings/cg_custom/cg`.

In [11]:
import site
import os
import shutil

# Localiza dónde se instaló kaggle-environments en el entorno de Colab
site_packages = site.getsitepackages()[0]
target_dir = os.path.join(site_packages, "kaggle_environments/envs/cabt/cg")

# Asumiendo que has subido tu carpeta completa de proyecto a Colab (o clonado de Github) al directorio actual (/content/)
source_dir = os.path.join(os.getcwd(), "remotethings/cg_custom/cg")

# Inyectar el motor
if os.path.exists(source_dir):
    if os.path.exists(target_dir):
        shutil.rmtree(target_dir)
    shutil.copytree(source_dir, target_dir)
    print("Motor customizado instalado exitosamente.")
else:
    print("ERROR: Asegúrate de haber subido los archivos del proyecto a Colab primero.")

Motor customizado instalado exitosamente.


## Iniciar el Entrenamiento
¡Todo listo! Ejecuta el script de entrenamiento. PPO correrá 10,000 pasos. Para subirlo a 1 millón, simplemente edita `src/rl/train.py` antes de correrlo.

In [31]:
!git stash
!git pull origin main --force

Saved working directory and index state WIP on main: 4ceffa1 Add safe stop try-except block
remote: Enumerating objects: 9, done.
remote: Counting objects: 100% (9/9), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 5 (delta 4), reused 5 (delta 4), pack-reused 0 (from 0)
Unpacking objects: 100% (5/5), 869 bytes | 869.00 KiB/s, done.
From https://github.com/chamakov/kaggle-pkm-agent
 * branch            main       -> FETCH_HEAD
   4ceffa1..ff7b653  main       -> origin/main
Updating 4ceffa1..ff7b653
Fast-forward
 src/rl/train.py | 26 ++++++++++++++++++++++----
 1 file changed, 22 insertions(+), 4 deletions(-)


In [ ]:
!python -u src/rl/train.py

[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO: Successfully loaded OpenSpiel environments: 31.
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_amazons
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_backgammon
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_python_ant_foraging
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_checkers
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_chess
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_clobber
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_coin_game
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_coin_game_arena
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_connect_four
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_dark_hex
[kaggle_e

In [ ]:
import json
import os
from stable_baselines3 import PPO
from src.rl.env_wrapper import CabtGymEnv

print("Cargando modelo PPO...")

# Construimos la ruta del archivo (nota: asegúrate de que se llame ppo_ no rppo_)
model_path = os.path.join(os.getcwd(), "best_models/best_model")

# ¡AQUÍ ESTÁ LA MAGIA! Usamos PPO.load
model = PPO.load(model_path)
env = CabtGymEnv(opponent_agent="random")

num_games = 50
wins = 0
invalid_moves = 0

print(f"Iniciando torneo de {num_games} partidas vs Random...")

for i in range(num_games):
    obs, info = env.reset()
    done = False

    while not done:
        action, _states = model.predict(obs, deterministic=True)
        obs, reward, done, truncated, info = env.step(action)

        if info.get("invalid_action"):
            invalid_moves += 1

    if reward > 0:
        wins += 1

    print(f"Partida {i+1}/{num_games} terminada. Recompensa final: {reward}")

win_rate = (wins / num_games) * 100
print("\n" + "="*40)
print(f"RESULTADOS DEL TORNEO")
print(f"Tasa de Victorias (Win Rate): {win_rate}%")
print(f"Total de movimientos inválidos cometidos: {invalid_moves}")
print("="*40)

replay_data = env.env.toJSON()
with open("replay_ppo_vs_random.json", "w") as f:
    json.dump(replay_data, f)

print("Última partida guardada como 'replay_ppo_vs_random.json'.")